# Apache Hudi - Schema Evolution

Apache Hudi supports schema evolution, which allows a table schema to change over time while keeping existing data readable.
This is important in real-world data lake pipelines where producers evolve event structures, add new fields, or publish newer versions of records.

## What you will learn

In this notebook, you will learn:

- What schema evolution means in Apache Hudi
- Creating an initial Hudi Copy-On-Write (COW) table
- Adding a new column to an existing Hudi table
- How Hudi handles missing values for older records
- Reading evolved table schema with Spark
- Verifying backward-compatible schema changes
- Understanding safe vs risky schema evolution patterns

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Spark Docker image under `/opt/spark/jars`.

Because of that, we do not use `spark.jars.packages` here. This avoids Ivy/Maven downloads from inside the notebook.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-05-Schema-Evolution")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.2


## Step 2 — Define shared paths and table name

This notebook is independent and creates its own table. It does not require any table from previous notebooks.

In [2]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_schema_evolution"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_schema_evolution
Table path: /workspace/data/hudi/riders_schema_evolution


## Step 3 — Clean previous run

For a tutorial notebook, we remove the previous table path so that reruns start from a known state.

In [3]:
import shutil
import os

if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print("No existing table path found.")

No existing table path found.


## Step 4 — Create initial data

The initial schema contains only `rider`, `city`, and `ts`.

Later we will evolve this schema by adding a new `rating` column.

In [4]:
from pyspark.sql.functions import to_timestamp

df_initial = spark.createDataFrame(
    [
        ("r1", "SFO", "2024-01-06 10:00:00"),
        ("r2", "NYC", "2024-01-06 11:00:00"),
        ("r3", "LAX", "2024-01-06 12:00:00"),
    ],
    ["rider", "city", "ts"]
).withColumn("ts", to_timestamp("ts"))

df_initial.show(truncate=False)
df_initial.printSchema()

[Stage 1:=======================================>                   (2 + 1) / 3]

+-----+----+-------------------+
|rider|city|ts                 |
+-----+----+-------------------+
|r1   |SFO |2024-01-06 10:00:00|
|r2   |NYC |2024-01-06 11:00:00|
|r3   |LAX |2024-01-06 12:00:00|
+-----+----+-------------------+

root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)



## Step 5 — Write the initial Hudi COW table

This creates a Copy-On-Write table with the initial schema.

The table uses:

- `rider` as the record key
- `city` as the partition path
- `ts` as the precombine field

In [5]:
hudi_write_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
    "hoodie.schema.update.enable": "true",
}

(
    df_initial.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print(f"Created initial Hudi table at: {TABLE_PATH}")

# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 35:>                                                         (0 + 3) / 3]

26/04/25 19:44:05 WARN HoodieTableFileSystemView: Partition: LAX is not available in store
26/04/25 19:44:05 WARN HoodieTableFileSystemView: Partition: NYC is not available in store
26/04/25 19:44:05 WARN HoodieTableFileSystemView: Partition: SFO is not available in store


Created initial Hudi table at: /workspace/data/hudi/riders_schema_evolution


## Step 6 — Read and inspect the initial schema

Before schema evolution, the table does not contain the `rating` column.

In [6]:
initial_read_df = spark.read.format("hudi").load(TABLE_PATH)

initial_read_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "rider",
    "city",
    "ts"
).orderBy("rider").show(truncate=False)

initial_read_df.printSchema()

+-------------------+------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_record_key|rider|city|ts                 |
+-------------------+------------------+-----+----+-------------------+
|20260425194346806  |r1                |r1   |SFO |2024-01-06 10:00:00|
|20260425194346806  |r2                |r2   |NYC |2024-01-06 11:00:00|
|20260425194346806  |r3                |r3   |LAX |2024-01-06 12:00:00|
+-------------------+------------------+-----+----+-------------------+

root
 |-- _hoodie_commit_time: string (nullable = true)
 |-- _hoodie_commit_seqno: string (nullable = true)
 |-- _hoodie_record_key: string (nullable = true)
 |-- _hoodie_partition_path: string (nullable = true)
 |-- _hoodie_file_name: string (nullable = true)
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)



## Step 7 — Evolve schema by adding a new column

Now we add a new `rating` column.

This is a backward-compatible schema change because old records can still be read. For old records, Spark/Hudi will return `NULL` for the new column.

In [7]:
df_evolve = spark.createDataFrame(
    [
        ("r4", "DEN", "2024-01-06 14:00:00", 4.5),
        ("r1", "SFO", "2024-01-06 15:00:00", 5.0),
    ],
    ["rider", "city", "ts", "rating"]
).withColumn("ts", to_timestamp("ts"))

df_evolve.show(truncate=False)
df_evolve.printSchema()

+-----+----+-------------------+------+
|rider|city|ts                 |rating|
+-----+----+-------------------+------+
|r4   |DEN |2024-01-06 14:00:00|4.5   |
|r1   |SFO |2024-01-06 15:00:00|5.0   |
+-----+----+-------------------+------+

root
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- rating: double (nullable = true)



## Step 8 — Write evolved data to the same table

The same Hudi table receives data with an additional column.

Because `hoodie.schema.update.enable` is enabled, Hudi can accept the evolved schema.

In [8]:
(
    df_evolve.write
    .format("hudi")
    .options(**hudi_write_options)
    .mode("append")
    .save(TABLE_PATH)
)

print("Schema evolution write completed.")

[Stage 49:>                                                         (0 + 2) / 2]

26/04/25 19:44:18 WARN HoodieTableFileSystemView: Partition: DEN is not available in store


26/04/25 19:44:19 WARN HoodieTableFileSystemView: Partition: DEN is not available in store
26/04/25 19:44:22 WARN HoodieTableFileSystemView: Partition: DEN is not available in store
Schema evolution write completed.


## Step 9 — Read evolved table

The table now contains the new `rating` column.

Older records that were written before the schema change will have `NULL` in `rating`.

In [9]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_evolved")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    rating,
    _hoodie_commit_time
FROM riders_evolved
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+------+-------------------+
|rider|city|ts                 |rating|_hoodie_commit_time|
+-----+----+-------------------+------+-------------------+
|r1   |SFO |2024-01-06 15:00:00|5.0   |20260425194413899  |
|r2   |NYC |2024-01-06 11:00:00|NULL  |20260425194346806  |
|r3   |LAX |2024-01-06 12:00:00|NULL  |20260425194346806  |
|r4   |DEN |2024-01-06 14:00:00|4.5   |20260425194413899  |
+-----+----+-------------------+------+-------------------+



## Step 10 — Verify the final schema

The final schema includes both the original columns and the newly added `rating` column.

In [10]:
evolved_df = spark.read.format("hudi").load(TABLE_PATH)

evolved_df.printSchema()

print("Final columns:")
for column_name in evolved_df.columns:
    print(column_name)

root
 |-- _hoodie_commit_time: string (nullable = true)
 |-- _hoodie_commit_seqno: string (nullable = true)
 |-- _hoodie_record_key: string (nullable = true)
 |-- _hoodie_partition_path: string (nullable = true)
 |-- _hoodie_file_name: string (nullable = true)
 |-- rider: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- rating: double (nullable = true)

Final columns:
_hoodie_commit_time
_hoodie_commit_seqno
_hoodie_record_key
_hoodie_partition_path
_hoodie_file_name
rider
city
ts
rating


## Step 11 — Inspect commit instants

We can inspect visible commit instants using the `_hoodie_commit_time` metadata column.

In [11]:
commits = [
    row["_hoodie_commit_time"]
    for row in evolved_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

print("Commit instants visible in the table:")
for commit in commits:
    print(commit)

Commit instants visible in the table:
20260425194346806
20260425194413899


## Step 12 — Safe and risky schema evolution patterns

Safe schema evolution patterns usually include:

- Adding nullable columns
- Adding columns with default handling in downstream logic
- Keeping existing column meanings stable

Risky schema evolution patterns include:

- Renaming columns without migration
- Changing column types directly
- Removing columns that downstream readers still expect
- Changing the meaning of an existing field

For production pipelines, schema changes should be tested with representative old and new records.

## Step 13 — Summary

In this notebook, you created an independent Hudi COW table and evolved its schema by adding a new column.

Key takeaways:

- Hudi can support backward-compatible schema changes.
- Adding a new nullable column is a common safe evolution pattern.
- Older records remain readable and return `NULL` for new fields.
- Schema evolution should be handled carefully in production pipelines.